# Feature Efficacy Analysis

This notebook evaluates the trading algorithm’s asset features plus three
additional research diagnostics: Frog-in-the-Pan (FIP), Bollinger bandwidth,
and FIP divided by Bollinger bandwidth. These additions are scoped to this
notebook only and do not change the trading algorithm’s routing contract.

Forward labels are used only for offline diagnostics; they must never be
passed into a trading policy or preprocessing fit. Run the cells from top to
bottom and edit the final configuration cell as needed.

## Install dependencies

Uncomment and run this cell in a fresh notebook environment such as Google Colab.

In [1]:
# %pip install -q yfinance pandas numpy scipy plotly scikit-learn

## Imports

In [2]:
from __future__ import annotations

import warnings
from dataclasses import dataclass, field
from typing import Callable, Literal

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import pearsonr, spearmanr
import plotly.express as px


CorrelationMethod = Literal["spearman", "pearson"]
FeatureFunction = Callable[[pd.DataFrame, "FeatureConfig"], pd.DataFrame]

## Config

Immutable settings and type aliases used throughout the analysis.

In [3]:
@dataclass(frozen=True)
class FeatureConfig:
    accumulation_window: int = 40

    klinger_fast_span: int = 34
    klinger_slow_span: int = 55
    klinger_signal_span: int = 13

    macd_fast_span: int = 12
    macd_slow_span: int = 26
    macd_signal_span: int = 9

    mr_ewma_span: int = 50
    mr_vol_window: int = 50
    momentum_window: int = 40
    momentum_quality_window: int = 40

    fip_window: int = 20
    bollinger_window: int = 20
    bollinger_std_multiplier: float = 2.0

    cmf_window: int = 20


@dataclass(frozen=True)
class PreprocessingConfig:
    rolling_window: int = 252
    scale: bool = True
    clip_lower: float | None = None
    clip_upper: float | None = None
    fill_null_value: float = 0.0


@dataclass(frozen=True)
class FeatureSpec:
    name: str
    output_columns: tuple[str, ...]
    function: FeatureFunction
    dependencies: tuple[str, ...] = field(default_factory=tuple)
    description: str = ""


DEFAULT_RETURN_HORIZONS = (5, 20, 60, 120)
DEFAULT_DRAWDOWN_HORIZONS = (5, 20, 60, 120)

## Data download

Download adjusted OHLCV data and normalize it to one row per ticker and date.

In [4]:
def download_ohlcv(
    tickers: list[str],
    start: str = "2015-01-01",
    end: str | None = None,
) -> pd.DataFrame:
    raw = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        group_by="column",
    )

    if raw.empty:
        raise ValueError("No data downloaded.")

    if isinstance(raw.columns, pd.MultiIndex):
        frames = []

        for ticker in tickers:
            if ticker not in raw["Close"].columns:
                warnings.warn(f"Ticker missing from yfinance output: {ticker}")
                continue

            df = pd.DataFrame(
                {
                    "date": raw.index,
                    "ticker": ticker,
                    "open": raw["Open"][ticker],
                    "high": raw["High"][ticker],
                    "low": raw["Low"][ticker],
                    "close": raw["Close"][ticker],
                    "volume": raw["Volume"][ticker],
                }
            )

            frames.append(df)

        if not frames:
            raise ValueError("No valid ticker data found in yfinance output.")

        out = pd.concat(frames, ignore_index=True)

    else:
        if len(tickers) != 1:
            raise ValueError("Expected MultiIndex columns for multiple tickers.")

        ticker = tickers[0]
        out = pd.DataFrame(
            {
                "date": raw.index,
                "ticker": ticker,
                "open": raw["Open"],
                "high": raw["High"],
                "low": raw["Low"],
                "close": raw["Close"],
                "volume": raw["Volume"],
            }
        )

    out["date"] = pd.to_datetime(out["date"])
    out = out.dropna(subset=["close", "volume"])
    out = out.sort_values(["ticker", "date"]).reset_index(drop=True)

    return out

## Feature helpers

Small validation and rolling-slope utilities shared by the feature functions.

In [5]:
def _rolling_normalized_slope(series: pd.Series, window: int) -> pd.Series:
    slope = (series - series.shift(window - 1)) / float(window - 1)
    variation = series.diff().rolling(window, min_periods=2).std()
    return slope / (variation + 1e-9)


def _require_columns(df: pd.DataFrame, required: set[str], context: str) -> None:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{context} missing required columns: {sorted(missing)}")

## Feature functions

These pandas implementations mirror `finrl.features.asset.compute_asset_features`
and produce only features used by the trading algorithm.

In [6]:
def add_base_features(df: pd.DataFrame, config: FeatureConfig) -> pd.DataFrame:
    del config
    _require_columns(df, {"close", "volume"}, "base features")
    close = df["close"].astype(float)
    df["return"] = close.pct_change()
    return df


def add_mean_reversion_features(df: pd.DataFrame, config: FeatureConfig) -> pd.DataFrame:
    _require_columns(df, {"close", "return"}, "mean-reversion features")
    close = df["close"].astype(float)
    ewma = close.ewm(span=config.mr_ewma_span, adjust=False).mean()
    realized_vol = df["return"].astype(float).rolling(config.mr_vol_window, min_periods=2).std()
    df["mr_ewma50_vol_gap"] = ((ewma - close) / (close + 1e-9)) / (realized_vol + 1e-9)
    df["ewma50_slope"] = _rolling_normalized_slope(np.log(ewma + 1e-9), config.accumulation_window)
    return df


def add_macd_features(df: pd.DataFrame, config: FeatureConfig) -> pd.DataFrame:
    _require_columns(df, {"close"}, "MACD features")
    close = df["close"].astype(float)
    fast = close.ewm(span=config.macd_fast_span, adjust=False).mean()
    slow = close.ewm(span=config.macd_slow_span, adjust=False).mean()
    signal = (fast - slow).ewm(span=config.macd_signal_span, adjust=False).mean()
    slope = _rolling_normalized_slope(signal, config.accumulation_window)
    df["acc_macd_signal"] = signal
    df["macd_signal_strength"] = signal * slope
    return df


def add_klinger_features(df: pd.DataFrame, config: FeatureConfig) -> pd.DataFrame:
    _require_columns(df, {"close", "volume"}, "Klinger features")
    signed_volume = df["volume"].astype(float) * np.sign(df["close"].astype(float).diff()).fillna(0.0)
    fast = signed_volume.ewm(span=config.klinger_fast_span, adjust=False).mean()
    slow = signed_volume.ewm(span=config.klinger_slow_span, adjust=False).mean()
    signal = (fast - slow).ewm(span=config.klinger_signal_span, adjust=False).mean()
    slope = _rolling_normalized_slope(signal, config.accumulation_window)
    df["acc_klinger_signal"] = signal
    df["klinger_signal_strength"] = signal * slope
    return df


def add_momentum_features(df: pd.DataFrame, config: FeatureConfig) -> pd.DataFrame:
    _require_columns(df, {"close", "return"}, "momentum features")
    close = df["close"].astype(float)
    returns = df["return"].astype(float)
    rolling_return = close / close.shift(config.momentum_window) - 1.0
    rolling_variance = returns.rolling(config.momentum_window, min_periods=2).var()
    df["acc_momentum_quality"] = rolling_return / (rolling_variance + 1e-9)
    return df


def add_cmf_features(df: pd.DataFrame, config: FeatureConfig) -> pd.DataFrame:
    _require_columns(df, {"high", "low", "close", "volume"}, "CMF features")
    high, low = df["high"].astype(float), df["low"].astype(float)
    close, volume = df["close"].astype(float), df["volume"].astype(float)
    multiplier = ((close - low) - (high - close)) / ((high - low) + 1e-9)
    cmf = (multiplier * volume).rolling(config.cmf_window, min_periods=2).sum() / (volume.rolling(config.cmf_window, min_periods=2).sum() + 1e-9)
    df["cmf"] = cmf
    df["cmf_slope"] = _rolling_normalized_slope(cmf, config.accumulation_window)
    cross_up = ((cmf.shift(1) <= 0.0) & (cmf > 0.0)).fillna(False)
    cross_down = ((cmf.shift(1) >= 0.0) & (cmf < 0.0)).fillna(False)
    df["cmf_cross_signal"] = cross_up.astype(float) - cross_down.astype(float)
    event = cross_up | cross_down
    group = event.cumsum()
    days_since = pd.Series(np.nan, index=df.index, dtype=float)
    for _, indices in df.groupby(group, sort=False).groups.items():
        if group.loc[indices].iloc[0] > 0:
            days_since.loc[indices] = np.arange(len(indices), dtype=float)
    df["cmf_days_since_cross"] = days_since
    return df

def add_frog_in_the_pan_features(df: pd.DataFrame, config: FeatureConfig) -> pd.DataFrame:
    """Add FIP, Bollinger bandwidth, and FIP normalized by bandwidth.

    FIP is the trailing signed-return persistence score: the sum of return
    signs divided by sqrt(window). Bollinger bandwidth is the relative width
    of a rolling mean +/- k standard-deviation envelope.
    """
    _require_columns(df, {"return", "close"}, "FIP/Bollinger features")
    returns = df["return"].astype(float)
    close = df["close"].astype(float)
    signed_returns = np.sign(returns).fillna(0.0)
    df["frog_in_the_pan"] = signed_returns.rolling(
        config.fip_window, min_periods=2
    ).sum() / np.sqrt(float(config.fip_window))
    middle = close.rolling(config.bollinger_window, min_periods=2).mean()
    std = close.rolling(config.bollinger_window, min_periods=2).std()
    df["bollinger_bandwidth"] = (
        2.0 * config.bollinger_std_multiplier * std / (middle.abs() + 1e-9)
    )
    df["fip_over_bollinger_bandwidth"] = df["frog_in_the_pan"] / (
        df["bollinger_bandwidth"] + 1e-9
    )
    return df

## Feature registry

Register feature families, their dependencies, and the explicit columns used in the efficacy study.

In [7]:
FEATURE_REGISTRY: dict[str, FeatureSpec] = {
    "base": FeatureSpec("base", ("return",), add_base_features, (), "Daily return."),
    "mean_reversion": FeatureSpec("mean_reversion", ("mr_ewma50_vol_gap", "ewma50_slope"), add_mean_reversion_features, ("base",), "EWMA-50 gap and slope."),
    "macd": FeatureSpec("macd", ("acc_macd_signal", "macd_signal_strength"), add_macd_features, (), "MACD signal and strength."),
    "klinger": FeatureSpec("klinger", ("acc_klinger_signal", "klinger_signal_strength"), add_klinger_features, (), "Klinger signal and strength."),
    "momentum": FeatureSpec("momentum", ("acc_momentum_quality",), add_momentum_features, ("base",), "Return scaled by variance."),
    "frog_in_the_pan": FeatureSpec(
        "frog_in_the_pan",
        ("frog_in_the_pan", "bollinger_bandwidth", "fip_over_bollinger_bandwidth"),
        add_frog_in_the_pan_features, ("base",),
        "Return-persistence, Bollinger width, and normalized FIP.",
    ),
    "cmf": FeatureSpec("cmf", ("cmf", "cmf_slope", "cmf_cross_signal", "cmf_days_since_cross"), add_cmf_features, (), "CMF level, slope, crossing, and crossing age."),
}

DEFAULT_ENABLED_FEATURES = ("mean_reversion", "macd", "klinger", "momentum", "cmf", "frog_in_the_pan")

DEFAULT_MODEL_FEATURES = (
    "mr_ewma50_vol_gap", "ewma50_slope", "acc_macd_signal", "acc_klinger_signal",
    "macd_signal_strength", "klinger_signal_strength", "acc_momentum_quality",
    "cmf", "cmf_slope", "cmf_cross_signal", "cmf_days_since_cross",
    "frog_in_the_pan", "bollinger_bandwidth", "fip_over_bollinger_bandwidth",
)

## Registry helpers

Resolve dependency order and validate that requested model columns are produced.

In [8]:
def resolve_feature_order(
    enabled_features: tuple[str, ...],
    registry: dict[str, FeatureSpec] = FEATURE_REGISTRY,
) -> tuple[str, ...]:
    """
    Expands dependencies and returns features in executable order.
    """
    resolved: list[str] = []
    visiting: set[str] = set()
    visited: set[str] = set()

    def visit(name: str) -> None:
        if name not in registry:
            raise KeyError(
                f"Unknown feature '{name}'. Available features: {sorted(registry)}"
            )

        if name in visited:
            return

        if name in visiting:
            raise ValueError(f"Circular feature dependency detected at '{name}'.")

        visiting.add(name)

        for dependency in registry[name].dependencies:
            visit(dependency)

        visiting.remove(name)
        visited.add(name)
        resolved.append(name)

    for feature_name in enabled_features:
        visit(feature_name)

    return tuple(dict.fromkeys(resolved))


def feature_output_columns(
    enabled_features: tuple[str, ...],
    registry: dict[str, FeatureSpec] = FEATURE_REGISTRY,
    include_dependency_outputs: bool = True,
) -> tuple[str, ...]:
    """
    Returns all output columns created by selected features.

    Usually this is useful for debugging, not necessarily for model input.
    For model input, pass explicit model_feature_columns.
    """
    feature_names = (
        resolve_feature_order(enabled_features, registry)
        if include_dependency_outputs
        else enabled_features
    )

    cols: list[str] = []

    for feature_name in feature_names:
        cols.extend(registry[feature_name].output_columns)

    return tuple(dict.fromkeys(cols))


def validate_model_feature_columns(
    available_columns: tuple[str, ...],
    model_feature_columns: tuple[str, ...],
) -> None:
    missing = set(model_feature_columns) - set(available_columns)
    if missing:
        raise ValueError(
            "model_feature_columns contains columns not produced by enabled features: "
            f"{sorted(missing)}"
        )

## Feature engineering pipeline

Run the selected feature families independently for each ticker.

In [9]:
def compute_features_for_one_ticker(
    df: pd.DataFrame,
    config: FeatureConfig,
    enabled_features: tuple[str, ...] = DEFAULT_ENABLED_FEATURES,
    registry: dict[str, FeatureSpec] = FEATURE_REGISTRY,
) -> pd.DataFrame:
    df = df.sort_values("date").copy()

    execution_order = resolve_feature_order(
        enabled_features=enabled_features,
        registry=registry,
    )

    for feature_name in execution_order:
        spec = registry[feature_name]
        df = spec.function(df, config)

    return df


def compute_asset_features(
    ohlcv: pd.DataFrame,
    config: FeatureConfig = FeatureConfig(),
    enabled_features: tuple[str, ...] = DEFAULT_ENABLED_FEATURES,
    registry: dict[str, FeatureSpec] = FEATURE_REGISTRY,
) -> pd.DataFrame:
    required = {"date", "ticker", "close", "volume", "high", "low"}
    _require_columns(ohlcv, required, "ohlcv")

    frames = []

    for _, group in ohlcv.groupby("ticker", sort=False):
        frames.append(
            compute_features_for_one_ticker(
                group,
                config=config,
                enabled_features=enabled_features,
                registry=registry,
            )
        )

    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values(["ticker", "date"]).reset_index(drop=True)

    return out

## Rolling preprocessing

Apply trailing-only cleaning and optional rolling z-score scaling per ticker.

In [10]:
def preprocess_features_for_one_ticker(
    df: pd.DataFrame,
    feature_columns: tuple[str, ...],
    config: PreprocessingConfig,
) -> pd.DataFrame:
    df = df.sort_values("date").copy()

    for col in feature_columns:
        x = df[col].astype(float).copy()

        if config.clip_lower is not None or config.clip_upper is not None:
            x = x.clip(lower=config.clip_lower, upper=config.clip_upper)

        x = x.replace([np.inf, -np.inf], np.nan)
        x = x.ffill().fillna(config.fill_null_value)

        if config.scale:
            rolling_mean = x.rolling(
                config.rolling_window,
                min_periods=1,
            ).mean()

            rolling_std = x.rolling(
                config.rolling_window,
                min_periods=2,
            ).std()

            z = (x - rolling_mean) / rolling_std
            z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0)
            df[col] = z
        else:
            df[col] = x

    return df


def preprocess_asset_features(
    features: pd.DataFrame,
    feature_columns: tuple[str, ...],
    config: PreprocessingConfig = PreprocessingConfig(),
) -> pd.DataFrame:
    required = {"date", "ticker", *feature_columns}
    _require_columns(features, required, "features")

    frames = []

    for _, group in features.groupby("ticker", sort=False):
        frames.append(
            preprocess_features_for_one_ticker(
                group,
                feature_columns=feature_columns,
                config=config,
            )
        )

    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values(["ticker", "date"]).reset_index(drop=True)

    return out

## Forward labels

Create future-return and future-drawdown labels for offline efficacy diagnostics only.

In [11]:
def make_forward_labels_for_one_ticker(
    df: pd.DataFrame,
    return_horizons: tuple[int, ...] = DEFAULT_RETURN_HORIZONS,
    drawdown_horizons: tuple[int, ...] = DEFAULT_DRAWDOWN_HORIZONS,
) -> pd.DataFrame:
    df = df.sort_values("date").copy()

    close = df["close"].astype(float)
    out = df[["date", "ticker"]].copy()

    for h in return_horizons:
        out[f"future_return_{h}d"] = close.shift(-h) / close - 1.0

    for h in drawdown_horizons:
        future_returns = []

        for step in range(1, h + 1):
            future_returns.append(close.shift(-step) / close - 1.0)

        future_returns_df = pd.concat(future_returns, axis=1)
        out[f"future_max_drawdown_{h}d"] = future_returns_df.min(axis=1)

    return out


def make_forward_labels(
    ohlcv: pd.DataFrame,
    return_horizons: tuple[int, ...] = DEFAULT_RETURN_HORIZONS,
    drawdown_horizons: tuple[int, ...] = DEFAULT_DRAWDOWN_HORIZONS,
) -> pd.DataFrame:
    _require_columns(ohlcv, {"date", "ticker", "close"}, "ohlcv")

    frames = []

    for _, group in ohlcv.groupby("ticker", sort=False):
        frames.append(
            make_forward_labels_for_one_ticker(
                group,
                return_horizons=return_horizons,
                drawdown_horizons=drawdown_horizons,
            )
        )

    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values(["ticker", "date"]).reset_index(drop=True)

    return out


def label_columns(
    return_horizons: tuple[int, ...] = DEFAULT_RETURN_HORIZONS,
    drawdown_horizons: tuple[int, ...] = DEFAULT_DRAWDOWN_HORIZONS,
) -> tuple[str, ...]:
    return tuple(
        [f"future_return_{h}d" for h in return_horizons]
        + [f"future_max_drawdown_{h}d" for h in drawdown_horizons]
    )

## Dataset construction

Join decision-date features to forward diagnostic labels and remove incomplete rows.

In [12]:
def build_feature_efficacy_dataset(
    preprocessed_features: pd.DataFrame,
    ohlcv: pd.DataFrame,
    feature_columns: tuple[str, ...],
    return_horizons: tuple[int, ...] = DEFAULT_RETURN_HORIZONS,
    drawdown_horizons: tuple[int, ...] = DEFAULT_DRAWDOWN_HORIZONS,
) -> pd.DataFrame:
    labels = make_forward_labels(
        ohlcv,
        return_horizons=return_horizons,
        drawdown_horizons=drawdown_horizons,
    )

    labels_cols = label_columns(
        return_horizons=return_horizons,
        drawdown_horizons=drawdown_horizons,
    )

    dataset = preprocessed_features[
        ["date", "ticker", "close", *feature_columns]
    ].merge(
        labels,
        on=["date", "ticker"],
        how="inner",
    )

    keep_cols = ["date", "ticker", "close", *feature_columns, *labels_cols]
    dataset = dataset[keep_cols].copy()

    numeric_cols = [*feature_columns, *labels_cols]
    dataset[numeric_cols] = dataset[numeric_cols].replace(
        [np.inf, -np.inf],
        np.nan,
    )

    dataset = dataset.dropna(subset=numeric_cols)

    return dataset.sort_values(["ticker", "date"]).reset_index(drop=True)

## Correlation analysis

Measure pooled feature-label rank or linear correlations.

In [13]:
def _safe_corr(
    x: pd.Series,
    y: pd.Series,
    method: CorrelationMethod = "spearman",
) -> float:
    valid = (
        x.replace([np.inf, -np.inf], np.nan).notna()
        & y.replace([np.inf, -np.inf], np.nan).notna()
    )

    x_valid = x[valid].astype(float)
    y_valid = y[valid].astype(float)

    if len(x_valid) < 3:
        return np.nan

    if x_valid.nunique() <= 1 or y_valid.nunique() <= 1:
        return np.nan

    if method == "spearman":
        return float(spearmanr(x_valid, y_valid).correlation)

    if method == "pearson":
        return float(pearsonr(x_valid, y_valid)[0])

    raise ValueError("method must be 'spearman' or 'pearson'")


def compute_feature_label_correlations(
    dataset: pd.DataFrame,
    feature_columns: tuple[str, ...],
    labels: tuple[str, ...] | None = None,
    method: CorrelationMethod = "spearman",
) -> pd.DataFrame:
    if labels is None:
        labels = tuple(
            col
            for col in dataset.columns
            if col.startswith("future_return_")
            or col.startswith("future_max_drawdown_")
        )

    rows = []

    for feature in feature_columns:
        for label in labels:
            corr = _safe_corr(dataset[feature], dataset[label], method=method)

            rows.append(
                {
                    "feature": feature,
                    "label": label,
                    "correlation": corr,
                    "abs_correlation": abs(corr) if np.isfinite(corr) else np.nan,
                }
            )

    out = pd.DataFrame(rows)
    out = out.sort_values("abs_correlation", ascending=False)

    return out.reset_index(drop=True)


def compute_correlation_matrix(
    dataset: pd.DataFrame,
    columns: tuple[str, ...],
    method: CorrelationMethod = "spearman",
) -> pd.DataFrame:
    if method == "spearman":
        return dataset[list(columns)].corr(method="spearman")

    if method == "pearson":
        return dataset[list(columns)].corr(method="pearson")

    raise ValueError("method must be 'spearman' or 'pearson'")

## Stability diagnostics

Repeat correlations by calendar year and summarize sign stability.

In [14]:
def compute_yearly_feature_label_correlations(
    dataset: pd.DataFrame,
    feature_columns: tuple[str, ...],
    labels: tuple[str, ...] | None = None,
    method: CorrelationMethod = "spearman",
) -> pd.DataFrame:
    if labels is None:
        labels = tuple(
            col
            for col in dataset.columns
            if col.startswith("future_return_")
            or col.startswith("future_max_drawdown_")
        )

    data = dataset.copy()
    data["year"] = pd.to_datetime(data["date"]).dt.year

    frames = []

    for year, group in data.groupby("year"):
        corr = compute_feature_label_correlations(
            group,
            feature_columns=feature_columns,
            labels=labels,
            method=method,
        )
        corr["year"] = year
        frames.append(corr)

    if not frames:
        return pd.DataFrame(
            columns=[
                "feature",
                "label",
                "correlation",
                "abs_correlation",
                "year",
            ]
        )

    return pd.concat(frames, ignore_index=True)


def summarize_yearly_stability(yearly_corr: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for (feature, label), group in yearly_corr.groupby(["feature", "label"]):
        vals = group["correlation"].replace([np.inf, -np.inf], np.nan).dropna()

        if len(vals) == 0:
            continue

        median_corr = vals.median()
        mean_corr = vals.mean()

        if median_corr == 0:
            sign_stability = np.nan
        else:
            expected_sign = np.sign(median_corr)
            sign_stability = (np.sign(vals) == expected_sign).mean()

        rows.append(
            {
                "feature": feature,
                "label": label,
                "median_corr": median_corr,
                "mean_corr": mean_corr,
                "abs_median_corr": abs(median_corr),
                "sign_stability": sign_stability,
                "years": len(vals),
            }
        )

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    return out.sort_values(
        ["abs_median_corr", "sign_stability"],
        ascending=[False, False],
    ).reset_index(drop=True)

## Partial correlation diagnostics

Estimate whether a feature adds rank information beyond selected controls.

In [15]:
def partial_spearman_corr(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    control_cols: list[str] | tuple[str, ...],
) -> float:
    """
    Spearman-style partial correlation: corr(x, y | controls).
    """
    from sklearn.linear_model import LinearRegression

    cols = [x_col, y_col, *control_cols]
    data = df[cols].replace([np.inf, -np.inf], np.nan).dropna()

    if len(data) < 10:
        return np.nan

    ranked = data.rank(method="average")

    controls = ranked[list(control_cols)].values
    x = ranked[x_col].values
    y = ranked[y_col].values

    x_model = LinearRegression().fit(controls, x)
    y_model = LinearRegression().fit(controls, y)

    x_resid = x - x_model.predict(controls)
    y_resid = y - y_model.predict(controls)

    return float(spearmanr(x_resid, y_resid).correlation)


def compare_raw_vs_partial_correlations(
    dataset: pd.DataFrame,
    features: tuple[str, ...],
    label: str = "future_return_60d",
    controls: tuple[str, ...] = ("mr_ewma50_vol_gap",),
) -> pd.DataFrame:
    rows = []

    for feature in features:
        raw_corr = _safe_corr(dataset[feature], dataset[label], method="spearman")

        partial_corr = partial_spearman_corr(
            dataset,
            x_col=feature,
            y_col=label,
            control_cols=controls,
        )

        rows.append(
            {
                "feature": feature,
                "label": label,
                "raw_spearman": raw_corr,
                "partial_spearman": partial_corr,
                "abs_drop": abs(raw_corr) - abs(partial_corr),
            }
        )

    return pd.DataFrame(rows).sort_values("abs_drop", ascending=False)

## Plotting

Interactive Plotly views of the strongest relationships and the full correlation matrix.

In [16]:
def plot_correlation_matrix(
    corr_matrix: pd.DataFrame,
    title: str = "Feature / Label Correlation Matrix",
) -> None:
    fig = px.imshow(
        corr_matrix,
        text_auto=".2f",
        aspect="auto",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        title=title,
    )
    fig.update_layout(
        width=max(1400, 42 * len(corr_matrix.columns)),
        height=max(1200, 42 * len(corr_matrix.index)),
        margin=dict(l=180, r=100, t=100, b=180),
    )

    fig.show()


def plot_feature_label_correlations(
    feature_label_corr: pd.DataFrame,
    top_n: int = 40,
    title: str = "Top Feature-Label Correlations",
) -> None:
    plot_df = feature_label_corr.head(top_n).copy()

    fig = px.bar(
        plot_df,
        x="correlation",
        y="feature",
        color="label",
        orientation="h",
        title=title,
    )

    fig.update_layout(
        yaxis={"categoryorder": "total ascending"},
        height=max(600, 22 * len(plot_df)),
    )

    fig.show()

## Full pipeline

Orchestrate data download, feature computation, preprocessing, labels, diagnostics, and plots.

In [17]:
def run_feature_efficacy_analysis(
    tickers: list[str],
    start: str = "2015-01-01",
    end: str | None = None,
    feature_config: FeatureConfig = FeatureConfig(),
    preprocessing_config: PreprocessingConfig = PreprocessingConfig(),
    enabled_features: tuple[str, ...] = DEFAULT_ENABLED_FEATURES,
    model_feature_columns: tuple[str, ...] = DEFAULT_MODEL_FEATURES,
    return_horizons: tuple[int, ...] = DEFAULT_RETURN_HORIZONS,
    drawdown_horizons: tuple[int, ...] = DEFAULT_DRAWDOWN_HORIZONS,
    method: CorrelationMethod = "spearman",
    plot: bool = True,
    yearly_stability: bool = True,
    partial_diagnostics: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    execution_order = resolve_feature_order(enabled_features)
    available_columns = feature_output_columns(enabled_features)
    validate_model_feature_columns(available_columns, model_feature_columns)

    print("Enabled features:", enabled_features)
    print("Execution order:", execution_order)
    print("Model feature columns:", model_feature_columns)

    print("\nDownloading OHLCV...")
    ohlcv = download_ohlcv(tickers=tickers, start=start, end=end)

    print("Computing features...")
    raw_features = compute_asset_features(
        ohlcv,
        config=feature_config,
        enabled_features=enabled_features,
    )

    print("Preprocessing model features...")
    preprocessed_features = preprocess_asset_features(
        raw_features,
        feature_columns=model_feature_columns,
        config=preprocessing_config,
    )

    print("Building future labels and supervised dataset...")
    dataset = build_feature_efficacy_dataset(
        preprocessed_features=preprocessed_features,
        ohlcv=ohlcv,
        feature_columns=model_feature_columns,
        return_horizons=return_horizons,
        drawdown_horizons=drawdown_horizons,
    )

    labels = label_columns(
        return_horizons=return_horizons,
        drawdown_horizons=drawdown_horizons,
    )

    print("Computing feature-label correlations...")
    feature_label_corr = compute_feature_label_correlations(
        dataset,
        feature_columns=model_feature_columns,
        labels=labels,
        method=method,
    )

    print("Computing full correlation matrix...")
    corr_matrix = compute_correlation_matrix(
        dataset,
        columns=model_feature_columns + labels,
        method=method,
    )

    print("\nDataset shape:", dataset.shape)

    print("\nTop feature-label correlations:")
    print(feature_label_corr.head(30))

    if yearly_stability:
        print("\nComputing yearly stability...")
        yearly_corr = compute_yearly_feature_label_correlations(
            dataset,
            feature_columns=model_feature_columns,
            labels=labels,
            method=method,
        )
        stability = summarize_yearly_stability(yearly_corr)

        print("\nTop yearly-stable feature-label correlations:")
        print(stability.head(30))

    if partial_diagnostics and "mr_ewma50_vol_gap" in model_feature_columns:
        print("\nRaw vs partial correlations controlling for mr_ewma50_vol_gap:")

        diagnostic_features = tuple(
            col for col in model_feature_columns if col != "mr_ewma50_vol_gap"
        )

        partial = compare_raw_vs_partial_correlations(
            dataset,
            features=diagnostic_features,
            label="future_return_60d",
            controls=("mr_ewma50_vol_gap",),
        )

        print(partial)

    if plot:
        plot_feature_label_correlations(
            feature_label_corr,
            top_n=40,
            title=f"Top Feature-Label Correlations ({method})",
        )

        plot_correlation_matrix(
            corr_matrix,
            title=f"Feature / Label Correlation Matrix ({method})",
        )

    return dataset, feature_label_corr, corr_matrix, preprocessed_features

## Run the analysis

Edit these values first. The default single-ticker run is intentionally small enough for an exploratory notebook session.

In [18]:
TICKERS = [
    "XLK"
]

# Trading features plus notebook-only research diagnostics.
ENABLED_FEATURES = DEFAULT_ENABLED_FEATURES
MODEL_FEATURE_COLUMNS = DEFAULT_MODEL_FEATURES

In [19]:
dataset, feature_label_corr, corr_matrix, preprocessed_features = run_feature_efficacy_analysis(
    tickers=TICKERS,
    start="2000-01-01",
    end="2026-01-01",
    enabled_features=ENABLED_FEATURES,
    model_feature_columns=MODEL_FEATURE_COLUMNS,
    method="spearman",
    plot=True,
    yearly_stability=True,
    partial_diagnostics=True,
)

Enabled features: ('mean_reversion', 'macd', 'klinger', 'momentum', 'cmf', 'frog_in_the_pan')
Execution order: ('base', 'mean_reversion', 'macd', 'klinger', 'momentum', 'cmf', 'frog_in_the_pan')
Model feature columns: ('mr_ewma50_vol_gap', 'ewma50_slope', 'acc_macd_signal', 'acc_klinger_signal', 'macd_signal_strength', 'klinger_signal_strength', 'acc_momentum_quality', 'cmf', 'cmf_slope', 'cmf_cross_signal', 'cmf_days_since_cross', 'frog_in_the_pan', 'bollinger_bandwidth', 'fip_over_bollinger_bandwidth')

Computing features...
Preprocessing model features...
Building future labels and supervised dataset...
Computing feature-label correlations...
Computing full correlation matrix...

Dataset shape: (6419, 25)

Top feature-label correlations:
                         feature                     label  correlation  \
0                frog_in_the_pan  future_max_drawdown_120d     0.111198   
1   fip_over_bollinger_bandwidth  future_max_drawdown_120d     0.105368   
2               cmf_cros

## Inspect the results

In [20]:
print(f"Dataset rows: {len(dataset):,}")
print(f"Preprocessed rows: {len(preprocessed_features):,}")
feature_label_corr.head(30)

Dataset rows: 6,419
Preprocessed rows: 6,539


,feature,label,correlation,abs_correlation
0,frog_in_the_pan,future_max_drawdown_120d,0.111198,0.111198
1,fip_over_bollinger_bandwidth,future_max_drawdown_120d,0.105368,0.105368
2,cmf_cross_signal,future_max_drawdown_20d,-0.099022,0.099022
3,cmf_cross_signal,future_max_drawdown_120d,-0.090903,0.090903
4,bollinger_bandwidth,future_max_drawdown_120d,-0.090379,0.090379
5,fip_over_bollinger_bandwidth,future_max_drawdown_20d,0.089225,0.089225
6,frog_in_the_pan,future_max_drawdown_20d,0.089167,0.089167
7,acc_macd_signal,future_return_60d,-0.087871,0.087871
8,mr_ewma50_vol_gap,future_return_60d,0.086724,0.086724
9,bollinger_bandwidth,future_max_drawdown_60d,-0.085929,0.085929
